# Extração Escalar Clínica com Expressões Regulares (Regex)
Notebook focado na definição, teste e consolidação dos padrões de expressões regulares para extração de atributos clínicos escalares do projeto **Diplotokens**.

Este notebook implementa como produto final a função modularizada `extract_clinical_attributes(text)` e valida seu comportamento através de asserções (`assert`) com exemplos reais extraídos de `data/raw/sample/cases.csv`.

In [1]:
from pathlib import Path
import re
import json
import pandas as pd
import diplo_grafo.config as cfg

# Carrega os dados de amostra
cases_df = pd.read_csv(cfg.SAMPLE_DATA_DIR / "cases.csv")
print(f"Total de casos carregados: {len(cases_df)}")
cases_df.head(2)

2026-09-08 21:13:25.515 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1


Total de casos carregados: 56


,article_id,age,case_id,case_text,gender
0,PMC5137649,44.0,PMC5137649_01,A 44-year-old woman presented with a 3-day his...,Female
1,PMC9387390,57.0,PMC9387390_01,A 57-year-old man with no significant past med...,Male


## 1. Catálogo de Padrões Regex
Definimos padrões com grupos de captura semânticos para cada tipo de atributo clínico:
- **Demografia:** Idade, gênero, peso de nascimento.
- **Sinais Vitais:** Pressão arterial, frequência cardíaca, frequência respiratória, saturação de oxigênio, temperatura.
- **Resultados de Laboratório:** Valores numéricos e unidades de medida (`mg/dL`, `ng/mL`, `iu/ml`, `U/L`, `mm/h`, `%`, etc.).
- **Dimensões & Medidas:** Lesões e órgãos em 3D, 2D e 1D, além de volumes.
- **Linha do Tempo & Duração:** Duração de sintomas, dias pós-operatórios (POD), intervalos.
- **Dosagens:** Doses farmacológicas de medicamentos.
- **Marcadores de Negação:** Termos como `denied`, `negative for`, `without`.

In [2]:
import re

PATTERNS = {
    # Demografia: Idade e genero (ex: "A 44-year-old woman", "65-year-old male", "20-month-old boy", "39-week-old newborn")
    "demographics": re.compile(
        r"(?:A|An)?\s*(\d{1,3})-(year|month|week|day)-old\s+(male|female|man|woman|boy|girl|newborn|infant)",
        re.IGNORECASE,
    ),
    # Peso neonatal/pediátrico (ex: "7-lb", "3.2 kg")
    "birth_weight": re.compile(
        r"\b(\d+(?:\.\d+)?)\s*(?:lb|lbs|kg|g)\b",
        re.IGNORECASE,
    ),
    # Pressão arterial (ex: "195/110 mmHg", "120/80", "136/74 mmHg")
    "blood_pressure": re.compile(
        r"\b(\d{2,3})/(\d{2,3})\s*(?:mmHg)?\b",
        re.IGNORECASE,
    ),
    # Frequência cardíaca (ex: "heart rate of 140", "pulse 77 bpm", "HR 92")
    "heart_rate": re.compile(
        r"\b(?:heart rate|pulse|HR|pulse rate)\s*(?:of|was|is)?\s*(\d{2,3})\s*(?:bpm|beats/min)?\b",
        re.IGNORECASE,
    ),
    # Frequência respiratória (ex: "respiratory rate of 30", "RR 18")
    "respiratory_rate": re.compile(
        r"\b(?:respiratory rate|RR)\s*(?:of|was|is)?\s*(\d{1,2})\s*(?:breaths/min)?\b",
        re.IGNORECASE,
    ),
    # Saturação de oxigênio (ex: "SpO2 98%", "saturation of 95%")
    "oxygen_saturation": re.compile(
        r"\b(?:SpO2|saturation|oxygen saturation)\s*(?:of|was|is)?\s*(\d{2,3})\s*%",
        re.IGNORECASE,
    ),
    # Temperatura corporal (ex: "105 degrees", "38.5 °C")
    "temperature": re.compile(
        r"\b(\d{2,3}(?:\.\d+)?)\s*(?:°C|°F|degrees)\b",
        re.IGNORECASE,
    ),
    # Dimensões 3D (ex: "9.5cm x 4.5cm x 2.0cm")
    "dimension_3d": re.compile(
        r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)",
        re.IGNORECASE,
    ),
    # Dimensões 2D (ex: "6cm x 9cm", "25 mm x 90 mm", "7cmx3cm")
    "dimension_2d": re.compile(
        r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)",
        re.IGNORECASE,
    ),
    # Dimensão 1D (ex: "6cm cystic lesion", "30 cm", "4 mm")
    "dimension_1d": re.compile(
        r"\b(\d+(?:\.\d+)?)\s*(cm|mm)\b",
        re.IGNORECASE,
    ),
    # Volumes (ex: "> 3 liters", "500 mL")
    "volume": re.compile(
        r"(?:>|<)?\s*(\d+(?:\.\d+)?)\s*(liters?|ml|mL)\b",
        re.IGNORECASE,
    ),
    # Valores de laboratório e marcadores com unidades
    # (ex: "12,476.5ng/ml", "1.9 mg/dL", "6iu/ml", "120 mm/h", "308 U/L", "2.4%")
    "lab_measurement": re.compile(
        r"(\d+(?:,\d+)*(?:\.\d+)?)\s*(mg/dL|g/dL|ng/mL|U/L|iu/ml|mm/h|mmol/L|g%|fL|/uL|x10\^9/L|K/uL|%)\b",
        re.IGNORECASE,
    ),
    # Linha do tempo: Dias de pós-operatório (ex: "postoperative day 4", "POD 7", "POD 11")
    "timeline_pod": re.compile(
        r"\b(?:POD|postoperative day)\s*(\d+)\b",
        re.IGNORECASE,
    ),
    # Duração e histórico temporal de sintomas (ex: "3-day history", "3-year history")
    "duration_history": re.compile(
        r"\b(\d+)-(day|week|month|year)\s+history\b",
        re.IGNORECASE,
    ),
    # Intervalos de acompanhamento (ex: "11 months after", "3 months later")
    "time_interval": re.compile(
        r"\b(\d+)\s+(day|week|month|year)s?\s+(?:after|later|prior|before|ago)\b",
        re.IGNORECASE,
    ),
    # Dosagens de medicamentos (ex: "75 mg", "40 mg", "300 mg", "2 mcg")
    "dosage": re.compile(
        r"\b(\d+(?:\.\d+)?)\s*(mg|mcg|g|IU|units)\b",
        re.IGNORECASE,
    ),
    # Marcadores de negação
    "negation": re.compile(
        r"\b(denied|denies|no evidence of|negative for|free of|unremarkable|without)\b",
        re.IGNORECASE,
    ),
}

## 2. Função de Extração Modularizada
A função `extract_clinical_attributes(text)` recebe qualquer texto ou sentença clínica e retorna um dicionário estruturado com todos os atributos encontrados, convertendo valores numéricos e salvando a posição exata (`span`).

In [3]:
def parse_number(num_str: str) -> float:
    """Converte string numérica (com suporte a vírgula de milhar) em float ou int."""
    cleaned = num_str.replace(",", "")
    val = float(cleaned)
    return int(val) if val.is_integer() else val


def extract_clinical_attributes(text: str) -> dict:
    """Extrai atributos clínicos escalares de um texto ou sentença."""
    results = {
        "demographics": [],
        "blood_pressure": [],
        "vital_signs": [],
        "dimensions": [],
        "lab_measurements": [],
        "timeline": [],
        "dosages": [],
        "negations": [],
    }

    # 1. Demografia
    for m in PATTERNS["demographics"].finditer(text):
        results["demographics"].append({
            "raw": m.group(0),
            "age": int(m.group(1)),
            "age_unit": m.group(2).lower(),
            "gender": m.group(3).lower(),
            "span": (m.start(), m.end()),
        })

    # 2. Pressão Arterial
    for m in PATTERNS["blood_pressure"].finditer(text):
        results["blood_pressure"].append({
            "raw": m.group(0),
            "systolic": int(m.group(1)),
            "diastolic": int(m.group(2)),
            "unit": "mmHg",
            "span": (m.start(), m.end()),
        })

    # 3. Sinais Vitais (FC, FR, SpO2, Temp)
    for m in PATTERNS["heart_rate"].finditer(text):
        results["vital_signs"].append({
            "type": "heart_rate",
            "raw": m.group(0),
            "value": int(m.group(1)),
            "unit": "bpm",
            "span": (m.start(), m.end()),
        })
    for m in PATTERNS["respiratory_rate"].finditer(text):
        results["vital_signs"].append({
            "type": "respiratory_rate",
            "raw": m.group(0),
            "value": int(m.group(1)),
            "unit": "breaths/min",
            "span": (m.start(), m.end()),
        })
    for m in PATTERNS["oxygen_saturation"].finditer(text):
        results["vital_signs"].append({
            "type": "oxygen_saturation",
            "raw": m.group(0),
            "value": int(m.group(1)),
            "unit": "%",
            "span": (m.start(), m.end()),
        })
    for m in PATTERNS["temperature"].finditer(text):
        results["vital_signs"].append({
            "type": "temperature",
            "raw": m.group(0),
            "value": parse_number(m.group(1)),
            "span": (m.start(), m.end()),
        })

    # 4. Dimensões (Priorizando 3D > 2D > 1D para evitar sobreposição)
    dimension_spans = set()
    for m in PATTERNS["dimension_3d"].finditer(text):
        results["dimensions"].append({
            "type": "3d",
            "raw": m.group(0),
            "values": [parse_number(m.group(1)), parse_number(m.group(3)), parse_number(m.group(5))],
            "unit": m.group(2).lower(),
            "span": (m.start(), m.end()),
        })
        for i in range(m.start(), m.end()):
            dimension_spans.add(i)

    for m in PATTERNS["dimension_2d"].finditer(text):
        if not any(i in dimension_spans for i in range(m.start(), m.end())):
            results["dimensions"].append({
                "type": "2d",
                "raw": m.group(0),
                "values": [parse_number(m.group(1)), parse_number(m.group(3))],
                "unit": m.group(2).lower(),
                "span": (m.start(), m.end()),
            })
            for i in range(m.start(), m.end()):
                dimension_spans.add(i)

    for m in PATTERNS["dimension_1d"].finditer(text):
        if not any(i in dimension_spans for i in range(m.start(), m.end())):
            results["dimensions"].append({
                "type": "1d",
                "raw": m.group(0),
                "value": parse_number(m.group(1)),
                "unit": m.group(2).lower(),
                "span": (m.start(), m.end()),
            })

    # 5. Medições de Laboratório
    for m in PATTERNS["lab_measurement"].finditer(text):
        results["lab_measurements"].append({
            "raw": m.group(0),
            "value": parse_number(m.group(1)),
            "unit": m.group(2),
            "span": (m.start(), m.end()),
        })

    # 6. Linha do Tempo e Duração
    for m in PATTERNS["timeline_pod"].finditer(text):
        results["timeline"].append({
            "type": "pod",
            "raw": m.group(0),
            "day": int(m.group(1)),
            "span": (m.start(), m.end()),
        })
    for m in PATTERNS["duration_history"].finditer(text):
        results["timeline"].append({
            "type": "symptom_history",
            "raw": m.group(0),
            "value": int(m.group(1)),
            "unit": m.group(2).lower(),
            "span": (m.start(), m.end()),
        })
    for m in PATTERNS["time_interval"].finditer(text):
        results["timeline"].append({
            "type": "interval",
            "raw": m.group(0),
            "value": int(m.group(1)),
            "unit": m.group(2).lower(),
            "span": (m.start(), m.end()),
        })

    # 7. Dosagens
    for m in PATTERNS["dosage"].finditer(text):
        results["dosages"].append({
            "raw": m.group(0),
            "value": parse_number(m.group(1)),
            "unit": m.group(2),
            "span": (m.start(), m.end()),
        })

    # 8. Negações
    for m in PATTERNS["negation"].finditer(text):
        results["negations"].append({
            "term": m.group(0).lower(),
            "span": (m.start(), m.end()),
        })

    return results

## 3. Casos de Teste e Validação
Executamos asserções formais (`assert`) contra frases reais do dataset `cases.csv` para validar cada classe de extração.

In [4]:
# ==========================================
# TESTE 1: Demografia e Histórico Temporal
# ==========================================
s1 = "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain."
res1 = extract_clinical_attributes(s1)

assert len(res1["demographics"]) == 1, "Deveria encontrar 1 registro demográfico"
assert res1["demographics"][0]["age"] == 44
assert res1["demographics"][0]["age_unit"] == "year"
assert res1["demographics"][0]["gender"] == "woman"

assert len(res1["timeline"]) == 1, "Deveria encontrar 1 histórico temporal"
assert res1["timeline"][0]["type"] == "symptom_history"
assert res1["timeline"][0]["value"] == 3
assert res1["timeline"][0]["unit"] == "day"
print("✓ Teste 1 (Demografia + Duração) passou com sucesso!")

# ==========================================
# TESTE 2: Dimensões 1D e 2D de Lesão
# ==========================================
s2 = "She underwent contrast enhanced CT, demonstrating a 6cm cystic lesion. Ultrasound revealed a cyst measuring 6cm x 9cm."
res2 = extract_clinical_attributes(s2)

assert len(res2["dimensions"]) == 2, f"Esperado 2 dimensões, obteve {len(res2['dimensions'])}"
dim_types = [d["type"] for d in res2["dimensions"]]
assert "1d" in dim_types and "2d" in dim_types
d2d = next(d for d in res2["dimensions"] if d["type"] == "2d")
assert d2d["values"] == [6, 9]
assert d2d["unit"] == "cm"
print("✓ Teste 2 (Dimensões 1D e 2D) passou com sucesso!")

# ==========================================
# TESTE 3: Valores Laboratoriais e Negações
# ==========================================
s3 = "FNA demonstrated no evidence of malignancy, CEA level of 12,476.5ng/ml and a CA 19-9 level of 6iu/ml."
res3 = extract_clinical_attributes(s3)

assert len(res3["negations"]) == 1
assert res3["negations"][0]["term"] == "no evidence of"

assert len(res3["lab_measurements"]) == 2
cea = res3["lab_measurements"][0]
assert cea["value"] == 12476.5
assert cea["unit"].lower() == "ng/ml"
ca19 = res3["lab_measurements"][1]
assert ca19["value"] == 6
assert ca19["unit"].lower() == "iu/ml"
print("✓ Teste 3 (Exames com unidades + Negação) passou com sucesso!")

# ==========================================
# TESTE 4: Dimensão 3D e Pós-Operatório (POD)
# ==========================================
s4 = "Final pathology revealed a 9.5cm x 4.5cm x 2.0cm cyst. The patient was discharged home on postoperative day 4 without complications."
res4 = extract_clinical_attributes(s4)

assert len(res4["dimensions"]) == 1
d3d = res4["dimensions"][0]
assert d3d["type"] == "3d"
assert d3d["values"] == [9.5, 4.5, 2.0]

assert len(res4["timeline"]) == 1
assert res4["timeline"][0]["type"] == "pod"
assert res4["timeline"][0]["day"] == 4

assert len(res4["negations"]) == 1
assert res4["negations"][0]["term"] == "without"
print("✓ Teste 4 (Dimensões 3D + POD) passou com sucesso!")

# ==========================================
# TESTE 5: Sinais Vitais e Pressão Arterial
# ==========================================
s5 = "On arrival, blood pressure was 136/74 mmHg, heart rate of 140 bpm, respiratory rate of 18, SpO2 98%, and temperature 105 degrees."
res5 = extract_clinical_attributes(s5)

assert len(res5["blood_pressure"]) == 1
assert res5["blood_pressure"][0]["systolic"] == 136
assert res5["blood_pressure"][0]["diastolic"] == 74

vital_map = {v["type"]: v["value"] for v in res5["vital_signs"]}
assert vital_map["heart_rate"] == 140
assert vital_map["respiratory_rate"] == 18
assert vital_map["oxygen_saturation"] == 98
assert vital_map["temperature"] == 105
print("✓ Teste 5 (Sinais Vitais Completos) passou com sucesso!")

# ==========================================
# TESTE 6: Dosagens Farmacológicas e Intervalo
# ==========================================
s6 = "The patient was started on 75 mg of clopidogrel and 40 mg of omeprazole. He presented 11 months after the procedure."
res6 = extract_clinical_attributes(s6)

assert len(res6["dosages"]) == 2
doses = [d["value"] for d in res6["dosages"]]
assert 75 in doses and 40 in doses

intervals = [t for t in res6["timeline"] if t["type"] == "interval"]
assert len(intervals) == 1
assert intervals[0]["value"] == 11
assert intervals[0]["unit"] == "month"
print("✓ Teste 6 (Dosagens e Intervalos) passou com sucesso!")

print("\nTODOS OS 6 CASOS DE TESTE PASSARAM COM SUCESSO!")

✓ Teste 1 (Demografia + Duração) passou com sucesso!
✓ Teste 2 (Dimensões 1D e 2D) passou com sucesso!
✓ Teste 3 (Exames com unidades + Negação) passou com sucesso!
✓ Teste 4 (Dimensões 3D + POD) passou com sucesso!
✓ Teste 5 (Sinais Vitais Completos) passou com sucesso!
✓ Teste 6 (Dosagens e Intervalos) passou com sucesso!

TODOS OS 6 CASOS DE TESTE PASSARAM COM SUCESSO!


## 4. Aplicação em Lote no Dataset (`cases.csv`)
Processamos os primeiros 10 casos clínicos completos e visualizamos a contagem de atributos extraídos em cada caso.

In [5]:
extracted_summary = []

for idx, row in cases_df.head(10).iterrows():
    text = row["case_text"]
    attrs = extract_clinical_attributes(text)
    
    extracted_summary.append({
        "case_id": row["case_id"],
        "demographics": len(attrs["demographics"]),
        "blood_pressure": len(attrs["blood_pressure"]),
        "vitals": len(attrs["vital_signs"]),
        "dimensions": len(attrs["dimensions"]),
        "lab_measurements": len(attrs["lab_measurements"]),
        "timeline": len(attrs["timeline"]),
        "dosages": len(attrs["dosages"]),
        "negations": len(attrs["negations"]),
    })

summary_df = pd.DataFrame(extracted_summary)
print("Resumo de Atributos Extraídos nos Primeiros 10 Casos:")
summary_df

Resumo de Atributos Extraídos nos Primeiros 10 Casos:


,case_id,demographics,blood_pressure,vitals,dimensions,lab_measurements,timeline,dosages,negations
0,PMC5137649_01,1,0,0,3,2,2,1,6
1,PMC9387390_01,1,0,0,2,0,8,0,2
2,PMC3437073_01,1,0,0,2,0,0,4,0
3,PMC7102447_01,1,0,0,2,3,0,0,0
4,PMC6354154_01,0,0,0,0,0,0,0,4
5,PMC4782471_01,1,1,2,4,2,1,3,0
6,PMC4835621_01,1,0,0,1,4,0,7,8
7,PMC8627357_01,0,0,0,0,0,0,0,3
8,PMC5833205_01,1,0,0,0,6,1,7,2
9,PMC3917415_01,0,0,0,0,3,0,5,1


## 5. Exemplo Detalhado do Caso 1 (`PMC5137649_01`)
Visualizando a estrutura completa dos atributos extraídos do primeiro caso clínico da amostra.

In [6]:
case1_text = cases_df.loc[0, "case_text"]
case1_attrs = extract_clinical_attributes(case1_text)

print(f"=== Caso: {cases_df.loc[0, 'case_id']} ===")
print(json.dumps(case1_attrs, indent=2))

=== Caso: PMC5137649_01 ===
{
  "demographics": [
    {
      "raw": "A 44-year-old woman",
      "age": 44,
      "age_unit": "year",
      "gender": "woman",
      "span": [
        0,
        19
      ]
    }
  ],
  "blood_pressure": [],
  "vital_signs": [],
  "dimensions": [
    {
      "type": "3d",
      "raw": "9.5cm x 4.5cm x 2.0cm",
      "values": [
        9.5,
        4.5,
        2
      ],
      "unit": "cm",
      "span": [
        2070,
        2091
      ]
    },
    {
      "type": "2d",
      "raw": "6cm x 9cm",
      "values": [
        6,
        9
      ],
      "unit": "cm",
      "span": [
        516,
        525
      ]
    },
    {
      "type": "1d",
      "raw": "6cm",
      "value": 6,
      "unit": "cm",
      "span": [
        337,
        340
      ]
    }
  ],
  "lab_measurements": [
    {
      "raw": "12,476.5ng/ml",
      "value": 12476.5,
      "unit": "ng/ml",
      "span": [
        777,
        790
      ]
    },
    {
      "raw": "6iu/ml",
   